<div style="background-color: #ffffff; color: #000000; padding: 30px;">
<img src="../media/images/kisz_logo.png" width="192" height="69" align="right" style="margin-right: 50px; margin-bottom: 50px;">
<h1>Time Series Analysis and Forecasting</h1>
</div>

<div style="background-color: #f6a800; color: #ffffff; padding: 10px;">
<h2>Solutions</h2>
<h2>Notebook B01: Exponential Smoothing Models</h2>
</div>

Worked solutions to the 3 exercises in
[Notebook B01: Exponential Smoothing Models](../notebooks/B01_Exponential_smoothing_models.ipynb).

**Try each exercise yourself first.** These notebooks are most useful as a check on your reasoning, and
least useful as something to read straight through. An exercise you attempted and got wrong teaches more
than a solution you agreed with.

Where an exercise asks a question rather than requesting code, the answer is written out under the code
that produces it. Several of them have answers that are more interesting than they look.

The setup cell below reproduces the state the exercises assume, so this notebook runs on its own.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="setup">Setup</h3>
</div>

In [ ]:
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.holtwinters import ExponentialSmoothing

sys.path.append("../notebooks")
import nb_config

sns.set_theme(style="whitegrid")

series = pd.read_parquet(nb_config.CDC_TEMP_PATH)["Brandenburg/Berlin"].asfreq("MS")

TEST_MONTHS = 24
SEASON_LENGTH = 12

train, test = series.iloc[:-TEST_MONTHS], series.iloc[-TEST_MONTHS:]


def mean_absolute_error(actual, forecast):
    return float(np.mean(np.abs(np.asarray(actual) - np.asarray(forecast))))


print(f"Train {len(train)} months, test {len(test)} months")

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="exercise-1">Exercise 1</h3>
</div>

> Fit Holt's method with `damped_trend=True` on the first 100 years of the series only (`series[:'1980']`) and forecast 240 months ahead. Plot it against the undamped version. How far apart are they after 20 years, and which looks more like a temperature series?

In [ ]:
early = series[:"1980"]
LONG_HORIZON = 240

undamped = ExponentialSmoothing(early, trend="add").fit()
damped = ExponentialSmoothing(early, trend="add", damped_trend=True).fit()

undamped_forecast = undamped.forecast(LONG_HORIZON)
damped_forecast = damped.forecast(LONG_HORIZON)

print(f"After 20 years (month 240):")
print(f"  undamped: {undamped_forecast.iloc[-1]:10.2f} °C")
print(f"  damped:   {damped_forecast.iloc[-1]:10.2f} °C")
print(f"  gap:      {abs(undamped_forecast.iloc[-1] - damped_forecast.iloc[-1]):10.2f} °C")
print()
print(f"For scale, the series has ranged from {series.min():.1f} to {series.max():.1f} °C")
print(f"and actually averaged {series['1981':'2000'].mean():.2f} °C over 1981-2000.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))

axes[0].plot(early["1970":], color="steelblue", linewidth=1.0, label="Train")
axes[0].plot(series["1981":"2000"], color="black", linewidth=1.0, label="What happened")
axes[0].plot(damped_forecast, color="seagreen", linewidth=1.5, linestyle="--", label="Damped")
axes[0].plot(undamped_forecast, color="crimson", linewidth=1.5, linestyle="--", label="Undamped")
axes[0].set_ylim(-40, 30)
axes[0].set_title("Zoomed to the range of the data", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Temperature (°C)")
axes[0].legend(fontsize=9, loc="lower left")

axes[1].plot(damped_forecast, color="seagreen", linewidth=1.5, label="Damped")
axes[1].plot(undamped_forecast, color="crimson", linewidth=1.5, label="Undamped")
axes[1].axhline(-273.15, color="black", linestyle=":", linewidth=1.2)
axes[1].text(damped_forecast.index[10], -250, "absolute zero", fontsize=9)
axes[1].set_title("Zoomed out far enough to see the undamped forecast",
                  fontsize=13, fontweight="bold")
axes[1].legend(fontsize=9)

for ax in axes:
    ax.tick_params(axis="x", rotation=30)
    ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

**They are about 690 °C apart, and the undamped forecast is not merely wrong — it is impossible.**

After twenty years the undamped model predicts **-701.67 °C**. Absolute zero is -273.15 °C, so this
forecast is more than four hundred degrees below the coldest temperature the universe permits. The damped
version predicts -9.85 °C, which is at least a temperature that Brandenburg has recorded.

Both are badly wrong — the region actually averaged 9.15 °C over 1981-2000 — but they are wrong in
qualitatively different ways, and that difference is the point of the exercise.

An undamped linear trend **compounds forever**. Holt fits a slope to the last stretch of data and adds it
once per step, so whatever slope happens to be current when the training data ends is projected out
indefinitely. Over twelve months that is a small extrapolation; over 240 it is a catastrophe. Nothing in
the model knows that temperature is a bounded physical quantity.

Damping multiplies the trend by $\phi < 1$ at each step, so the projected slope decays geometrically and
the forecast flattens to a finite asymptote. It is still an extrapolation, and it is still wrong here, but
it fails gracefully.

The practical rule follows directly: **damp any trend you intend to project further than a few periods.**
The longer the horizon, the less a slope estimated from recent data deserves to be trusted, and damping is
the cheapest possible way of encoding that.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="exercise-2">Exercise 2</h3>
</div>

> Take the Rossmann store sales from Notebook A04, aggregate to weekly totals for a single store, and fit both variants with `seasonal_periods=52`. Which does AIC prefer? Does the choice matter as much as picking the right seasonal period?

In [ ]:
sales = pd.read_csv(nb_config.ROSSMANN_TRAIN_PATH, parse_dates=["Date"], low_memory=False)
store = sales[sales["Store"] == 1].set_index("Date").sort_index()

weekly = store["Sales"].resample("W").sum()
weekly = weekly[weekly > 0]

HOLDOUT_WEEKS = 8
weekly_train, weekly_test = weekly.iloc[:-HOLDOUT_WEEKS], weekly.iloc[-HOLDOUT_WEEKS:]

print(f"{len(weekly)} weeks of data = {len(weekly) / 52:.1f} yearly cycles")
print(f"Fitting 52 seasonal factors from {len(weekly_train)} observations")

In [ ]:
candidates = {
    "additive, m=52": dict(trend="add", seasonal="add", seasonal_periods=52),
    "multiplicative, m=52": dict(trend="add", seasonal="mul", seasonal_periods=52),
    "additive, m=13": dict(trend="add", seasonal="add", seasonal_periods=13),
    "additive, m=4": dict(trend="add", seasonal="add", seasonal_periods=4),
    "no seasonality": dict(trend="add"),
}

rows = []
for name, settings in candidates.items():
    fitted = ExponentialSmoothing(weekly_train, **settings).fit()
    rows.append({
        "model": name,
        "parameters": len(fitted.params_formatted),
        "AIC": fitted.aic,
        "Test MAE": mean_absolute_error(weekly_test, fitted.forecast(HOLDOUT_WEEKS)),
    })

pd.DataFrame(rows).set_index("model").round(1)

**AIC prefers the multiplicative variant**, 2112 against 2132, and the held-out error agrees: 1436
against 1734.

To the second question — **no, the choice of variant matters considerably less than the choice of
period.** Compare the gaps:

- additive versus multiplicative at m = 52: about **20 AIC points**, and 300 in held-out MAE.
- m = 52 versus no seasonality at all: about **37 AIC points**, and 1,000 in held-out MAE.

Getting the period wrong is worse still. The m = 13 and m = 4 models are *worse than no seasonality*,
both on AIC and on held-out error, because they impose a cycle that is not there and spend parameters
doing it. A yearly rhythm in retail sales exists; a quarterly one does not, and asserting it costs
accuracy.

One caveat deserves stating plainly, because the numbers look more convincing than the evidence warrants.
**135 weeks is 2.6 yearly cycles, and we are estimating 52 seasonal factors from it** — roughly two and a
half observations per factor. That is very thin. The held-out improvement is real, and it is driven mostly
by the Christmas period, which is the one part of the yearly pattern that repeats strongly enough to be
learned from two examples. With a longer history the estimate would be far more trustworthy, and with a
shorter one this approach would simply be memorising noise.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="exercise-3">Exercise 3</h3>
</div>

> Run the rolling-origin evaluation from Notebook A06 on Holt-Winters and the seasonal naive forecast, with `horizon=12` over 10 origins. Does Holt-Winters beat the baseline at every origin, or only on average? Given the spread you found in A06, how confident are you in the 30% improvement reported above?

In [ ]:
HORIZON = 12
N_ORIGINS = 10
STEP = 12

rows = []
for k in range(N_ORIGINS):
    end = len(series) - HORIZON - (N_ORIGINS - 1 - k) * STEP
    history, actual = series.iloc[:end], series.iloc[end:end + HORIZON]

    holt_winters = ExponentialSmoothing(
        history, trend="add", seasonal="add", seasonal_periods=SEASON_LENGTH
    ).fit()

    cycle = history.iloc[-SEASON_LENGTH:].to_numpy()

    rows.append({
        "origin": history.index[-1].date(),
        "Holt-Winters": mean_absolute_error(actual, holt_winters.forecast(HORIZON)),
        "Seasonal naive": mean_absolute_error(
            actual, [cycle[i % SEASON_LENGTH] for i in range(HORIZON)]
        ),
    })

by_origin = pd.DataFrame(rows).set_index("origin")
by_origin["HW wins"] = by_origin["Holt-Winters"] < by_origin["Seasonal naive"]

by_origin.round(2)

In [ ]:
summary = by_origin[["Holt-Winters", "Seasonal naive"]].agg(["mean", "std", "min", "max"])

print(summary.round(2).to_string())
print()
print(f"Holt-Winters wins at {int(by_origin['HW wins'].sum())} of {N_ORIGINS} origins")
print(f"Improvement on the average: "
      f"{1 - by_origin['Holt-Winters'].mean() / by_origin['Seasonal naive'].mean():.1%}")

**Only on average: Holt-Winters wins at 8 of the 10 origins, not all of them.** It loses in 2019 (1.32
against 1.21) and again in 2022 (1.11 against 1.03), both times narrowly.

On the second question, the evidence is better than a single split but the answer has two parts.

**The 30% figure holds up remarkably well.** Averaged across ten origins, Holt-Winters scores 1.32 against
the baseline's 1.90, an improvement of **30.3%** — almost exactly the 30% the notebook reported from one
split. That agreement is not guaranteed and it is reassuring: the headline number was not an artefact of a
lucky test window.

**But the per-origin spread is large enough to swallow the difference.** Holt-Winters ranges from 0.97 to
2.01 across origins and the baseline from 1.03 to 3.08. Those ranges overlap substantially, and at two
origins the ordering reverses. So the honest statement is not "Holt-Winters is 30% better" but "**on
average across a decade Holt-Winters is about 30% better, and in any given year it is usually but not
always ahead**".

That distinction matters for how you would use the model. If you forecast once a year and are judged on
that one forecast, a 20% chance of being beaten by a one-line baseline is a real risk to describe. If you
forecast repeatedly and are judged on the average, the 30% is what you get. Notebook
[A06](../notebooks/A06_Evaluating_models.ipynb) argued for measuring the spread; this is what it is
for.

---

Back to [Notebook B01](../notebooks/B01_Exponential_smoothing_models.ipynb), or on to
[Notebook B02](../notebooks/B02_ARIMA_models.ipynb).